In [1]:
# Cellule 1 - Import libraries for feature engineering
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import os

print("Libraries imported successfully")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

Libraries imported successfully
Pandas version: 2.3.3
NumPy version: 2.2.6


In [2]:
# Cellule 2 - Charger les données préparées
import os

# Chemins des fichiers
movies_path = '../data/processed/movies_enriched.csv'
ratings_path = '../data/raw/ratings.csv'
user_stats_path = '../user_stats.csv'
movie_stats_path = '../movie_stats.csv'

# Chargement
print("Chargement des fichiers...")
df_movies = pd.read_csv(movies_path)
df_ratings = pd.read_csv(ratings_path)
df_user_stats = pd.read_csv(user_stats_path)
df_movie_stats = pd.read_csv(movie_stats_path)

print(f"\n Films enrichis: {len(df_movies)} lignes")
print(f" Notes brutes: {len(df_ratings)} lignes")
print(f" Statistiques utilisateurs: {len(df_user_stats)} lignes")
print(f" Statistiques films: {len(df_movie_stats)} lignes")

print("\nAperçu des films enrichis:")
display(df_movies.head())

Chargement des fichiers...

 Films enrichis: 9742 lignes
 Notes brutes: 100836 lignes
 Statistiques utilisateurs: 610 lignes
 Statistiques films: 9724 lignes

Aperçu des films enrichis:


,movieId,title,genres,genre_array,genre_count,year_str,rating_count,avg_rating,std_rating,min_rating,max_rating
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,"['Adventure', 'Animation', 'Children', 'Comedy...",5,1995.0,215,3.920930,0.834859,0.5,5.0
1,2,Jumanji (1995),Adventure|Children|Fantasy,"['Adventure', 'Children', 'Fantasy']",3,1995.0,110,3.431818,0.881713,0.5,5.0
2,3,Grumpier Old Men (1995),Comedy|Romance,"['Comedy', 'Romance']",2,1995.0,52,3.259615,1.054823,0.5,5.0
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,"['Comedy', 'Drama', 'Romance']",3,1995.0,7,2.357143,0.852168,1.0,3.0
4,5,Father of the Bride Part II (1995),Comedy,['Comedy'],1,1995.0,49,3.071429,0.907148,0.5,5.0


In [3]:
# Cellule 3 - Quick data exploration

print("=== MOVIES INFORMATION ===")
print(f"Available columns: {list(df_movies.columns)}")
print(f"Data types:")
print(df_movies.dtypes)
print(f"\nMissing values:")
print(df_movies.isnull().sum())

print("\n=== RATINGS INFORMATION ===")
print(f"Available columns: {list(df_ratings.columns)}")
print(f"Data types:")
print(df_ratings.dtypes)
print(f"\nMissing values:")
print(df_ratings.isnull().sum())

print("\n=== RATINGS STATISTICS ===")
print(df_ratings['rating'].describe())

=== MOVIES INFORMATION ===
Available columns: ['movieId', 'title', 'genres', 'genre_array', 'genre_count', 'year_str', 'rating_count', 'avg_rating', 'std_rating', 'min_rating', 'max_rating']
Data types:
movieId           int64
title            object
genres           object
genre_array      object
genre_count       int64
year_str        float64
rating_count      int64
avg_rating      float64
std_rating      float64
min_rating      float64
max_rating      float64
dtype: object

Missing values:
movieId          0
title            0
genres           0
genre_array      0
genre_count      0
year_str        13
rating_count     0
avg_rating       0
std_rating       0
min_rating       0
max_rating       0
dtype: int64

=== RATINGS INFORMATION ===
Available columns: ['userId', 'movieId', 'rating', 'timestamp']
Data types:
userId         int64
movieId        int64
rating       float64
timestamp      int64
dtype: object

Missing values:
userId       0
movieId      0
rating       0
timestamp    0


In [4]:
# Cellule 4 - Créer la matrice utilisateurs-films (table pivot)
# Cette matrice sera utilisée pour les modèles de recommandation

print("Creating user-movie matrix...")

# Créer une table pivot : utilisateurs en lignes, films en colonnes, notes en valeurs
user_movie_matrix = df_ratings.pivot_table(
    index='userId', 
    columns='movieId', 
    values='rating'
)

print(f"Matrix shape: {user_movie_matrix.shape[0]} users × {user_movie_matrix.shape[1]} movies")
print(f"Fill rate: {(~user_movie_matrix.isnull()).sum().sum() / (user_movie_matrix.shape[0] * user_movie_matrix.shape[1]) * 100:.2f}%")

print("\nMatrix preview (first 10 users, first 10 movies):")
display(user_movie_matrix.iloc[:10, :10])

Creating user-movie matrix...
Matrix shape: 610 users × 9724 movies
Fill rate: 1.70%

Matrix preview (first 10 users, first 10 movies):


movieId,1,2,3,4,5,6,7,8,9,10
userId,,,,,,,,,,
1,4.0,NaN,4.0,NaN,NaN,4.0,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,4.0,5.0,3.0,5.0,4.0,4.0,3.0,NaN,3.0
7,4.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# Cellule 5 - Normaliser les notes (centrer par utilisateur)
# Pour éviter les biais (utilisateurs généreux vs sévères)

print("Normalizing ratings...")

# Créer une copie de la matrice
user_movie_norm = user_movie_matrix.copy()

# Pour chaque utilisateur, soustraire sa moyenne personnelle
user_means = user_movie_norm.mean(axis=1)
user_movie_norm = user_movie_norm.sub(user_means, axis=0)

print(f"Normalized matrix shape: {user_movie_norm.shape[0]} users × {user_movie_norm.shape[1]} movies")

# Afficher un exemple
print("\nExample - User 1 ratings:")
print("Original ratings:")
print(user_movie_matrix.loc[1].dropna().head())
print(f"User 1 mean: {user_means.loc[1]:.2f}")
print("\nNormalized ratings (rating - user_mean):")
print(user_movie_norm.loc[1].dropna().head())

Normalizing ratings...
Normalized matrix shape: 610 users × 9724 movies

Example - User 1 ratings:
Original ratings:
movieId
1     4.0
3     4.0
6     4.0
47    5.0
50    5.0
Name: 1, dtype: float64
User 1 mean: 4.37

Normalized ratings (rating - user_mean):
movieId
1    -0.366379
3    -0.366379
6    -0.366379
47    0.633621
50    0.633621
Name: 1, dtype: float64


In [6]:
# Cellule 6 - One-hot encoding des genres
# But: Transformer les genres (texte) en colonnes numériques

print("Creating genre features...")

# Séparer les genres (déjà fait dans Phase 3, mais on va créer des colonnes binaires)
from sklearn.preprocessing import MultiLabelBinarizer

# Convertir les chaînes de genres en listes
df_movies['genres_list'] = df_movies['genres'].str.split('|')

# Créer le one-hot encoding
mlb = MultiLabelBinarizer()
genre_encoded = mlb.fit_transform(df_movies['genres_list'])
genre_df = pd.DataFrame(genre_encoded, columns=mlb.classes_, index=df_movies.index)

print(f"Genres trouvés: {list(genre_df.columns)}")
print(f"Shape: {genre_df.shape}")

# Ajouter au DataFrame principal
df_movies = pd.concat([df_movies, genre_df], axis=1)

print("\nAperçu des colonnes de genres ajoutées:")
display(df_movies[['movieId', 'title'] + list(genre_df.columns[:5])].head())

Creating genre features...
Genres trouvés: ['(no genres listed)', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
Shape: (9742, 20)

Aperçu des colonnes de genres ajoutées:


,movieId,title,(no genres listed),Action,Adventure,Animation,Children
0,1,Toy Story (1995),0,0,1,1,1
1,2,Jumanji (1995),0,0,1,0,1
2,3,Grumpier Old Men (1995),0,0,0,0,0
3,4,Waiting to Exhale (1995),0,0,0,0,0
4,5,Father of the Bride Part II (1995),0,0,0,0,0


In [7]:
# Cellule 7 - Création du DataFrame final pour les films
# But: Rassembler toutes les features des films pour les modèles

print("Creating final movie features dataframe...")

# Sélectionner les colonnes utiles
movie_features = df_movies[[
    'movieId', 'title', 'year_str', 'genre_count', 
    'rating_count', 'avg_rating', 'std_rating'
]].copy()

# Ajouter les colonnes de genres (déjà encodées)
genre_cols = list(genre_df.columns)
for col in genre_cols:
    movie_features[col] = df_movies[col].values

print(f"Shape du DataFrame films: {movie_features.shape}")
print(f"Colonnes: {list(movie_features.columns)}")

print("\nAperçu des features films:")
display(movie_features.head())

print("\nStatistiques des features numériques:")
display(movie_features.describe())

Creating final movie features dataframe...
Shape du DataFrame films: (9742, 27)
Colonnes: ['movieId', 'title', 'year_str', 'genre_count', 'rating_count', 'avg_rating', 'std_rating', '(no genres listed)', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']

Aperçu des features films:


,movieId,title,year_str,genre_count,rating_count,avg_rating,std_rating,(no genres listed),Action,Adventure,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),1995.0,5,215,3.920930,0.834859,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,2,Jumanji (1995),1995.0,3,110,3.431818,0.881713,0,0,1,...,0,0,0,0,0,0,0,0,0,0
2,3,Grumpier Old Men (1995),1995.0,2,52,3.259615,1.054823,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,4,Waiting to Exhale (1995),1995.0,3,7,2.357143,0.852168,0,0,0,...,0,0,0,0,0,1,0,0,0,0
4,5,Father of the Bride Part II (1995),1995.0,1,49,3.071429,0.907148,0,0,0,...,0,0,0,0,0,0,0,0,0,0



Statistiques des features numériques:


,movieId,year_str,genre_count,rating_count,avg_rating,std_rating,(no genres listed),Action,Adventure,Animation,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
count,9742.000000,9729.000000,9742.000000,9742.000000,9742.000000,9742.000000,9742.000000,9742.000000,9742.000000,9742.000000,...,9742.000000,9742.000000,9742.000000,9742.000000,9742.000000,9742.000000,9742.000000,9742.000000,9742.000000,9742.000000
mean,42200.353623,1994.613629,2.263396,10.350647,3.256420,0.540693,0.003490,0.187641,0.129645,0.062718,...,0.008930,0.100390,0.016218,0.034285,0.058817,0.163827,0.100595,0.194416,0.039212,0.017142
std,52160.494854,18.535219,1.128720,22.384729,0.880292,0.509959,0.058976,0.390445,0.335930,0.242468,...,0.094083,0.300535,0.126321,0.181968,0.235295,0.370137,0.300808,0.395771,0.194108,0.129808
min,1.000000,1902.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,3248.250000,1988.000000,1.000000,1.000000,2.785714,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,7300.000000,1999.000000,2.000000,3.000000,3.416667,0.577350,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,76232.000000,2008.000000,3.000000,9.000000,3.909091,0.927806,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,193609.000000,2018.000000,10.000000,329.000000,5.000000,3.181981,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [8]:
# Cellule 7 - Création du DataFrame final pour les films (CORRIGÉ)
# But: Rassembler toutes les features des films pour les modèles

print("Creating final movie features dataframe...")

# Sélectionner les colonnes utiles
movie_features = df_movies[[
    'movieId', 'title', 'year_str', 'genre_count', 
    'rating_count', 'avg_rating', 'std_rating'
]].copy()

# Ajouter les colonnes de genres (déjà encodées)
genre_cols = list(genre_df.columns)
for col in genre_cols:
    movie_features[col] = df_movies[col].values.flatten()  # .flatten() garantit une forme 1D

print(f"Shape du DataFrame films: {movie_features.shape}")
print(f"Colonnes: {list(movie_features.columns)}")

print("\nAperçu des features films:")
display(movie_features.head())

print("\nStatistiques des features numériques:")
display(movie_features.describe())

Creating final movie features dataframe...
Shape du DataFrame films: (9742, 27)
Colonnes: ['movieId', 'title', 'year_str', 'genre_count', 'rating_count', 'avg_rating', 'std_rating', '(no genres listed)', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']

Aperçu des features films:


,movieId,title,year_str,genre_count,rating_count,avg_rating,std_rating,(no genres listed),Action,Adventure,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),1995.0,5,215,3.920930,0.834859,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,2,Jumanji (1995),1995.0,3,110,3.431818,0.881713,0,0,1,...,0,0,0,0,0,0,0,0,0,0
2,3,Grumpier Old Men (1995),1995.0,2,52,3.259615,1.054823,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,4,Waiting to Exhale (1995),1995.0,3,7,2.357143,0.852168,0,0,0,...,0,0,0,0,0,1,0,0,0,0
4,5,Father of the Bride Part II (1995),1995.0,1,49,3.071429,0.907148,0,0,0,...,0,0,0,0,0,0,0,0,0,0



Statistiques des features numériques:


,movieId,year_str,genre_count,rating_count,avg_rating,std_rating,(no genres listed),Action,Adventure,Animation,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
count,9742.000000,9729.000000,9742.000000,9742.000000,9742.000000,9742.000000,9742.000000,9742.000000,9742.000000,9742.000000,...,9742.000000,9742.000000,9742.000000,9742.000000,9742.000000,9742.000000,9742.000000,9742.000000,9742.000000,9742.000000
mean,42200.353623,1994.613629,2.263396,10.350647,3.256420,0.540693,0.003490,0.187641,0.129645,0.062718,...,0.008930,0.100390,0.016218,0.034285,0.058817,0.163827,0.100595,0.194416,0.039212,0.017142
std,52160.494854,18.535219,1.128720,22.384729,0.880292,0.509959,0.058976,0.390445,0.335930,0.242468,...,0.094083,0.300535,0.126321,0.181968,0.235295,0.370137,0.300808,0.395771,0.194108,0.129808
min,1.000000,1902.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,3248.250000,1988.000000,1.000000,1.000000,2.785714,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,7300.000000,1999.000000,2.000000,3.000000,3.416667,0.577350,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,76232.000000,2008.000000,3.000000,9.000000,3.909091,0.927806,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,193609.000000,2018.000000,10.000000,329.000000,5.000000,3.181981,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [9]:
# Cellule 7 - Création du DataFrame final pour les films (CORRIGÉ V2)
# But: Rassembler toutes les features des films pour les modèles

print("Creating final movie features dataframe...")

# Sélectionner les colonnes utiles
movie_features = df_movies[[
    'movieId', 'title', 'year_str', 'genre_count', 
    'rating_count', 'avg_rating', 'std_rating'
]].copy()

# Ajouter les colonnes de genres (déjà encodées)
genre_cols = list(genre_df.columns)
for col in genre_cols:
    # Utiliser squeeze() pour garantir un format 1D
    movie_features[col] = df_movies[col].squeeze()

print(f"Shape du DataFrame films: {movie_features.shape}")
print(f"Colonnes: {list(movie_features.columns)}")

print("\nAperçu des features films:")
display(movie_features.head())

print("\nStatistiques des features numériques:")
display(movie_features.describe())


Creating final movie features dataframe...
Shape du DataFrame films: (9742, 27)
Colonnes: ['movieId', 'title', 'year_str', 'genre_count', 'rating_count', 'avg_rating', 'std_rating', '(no genres listed)', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']

Aperçu des features films:


,movieId,title,year_str,genre_count,rating_count,avg_rating,std_rating,(no genres listed),Action,Adventure,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),1995.0,5,215,3.920930,0.834859,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,2,Jumanji (1995),1995.0,3,110,3.431818,0.881713,0,0,1,...,0,0,0,0,0,0,0,0,0,0
2,3,Grumpier Old Men (1995),1995.0,2,52,3.259615,1.054823,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,4,Waiting to Exhale (1995),1995.0,3,7,2.357143,0.852168,0,0,0,...,0,0,0,0,0,1,0,0,0,0
4,5,Father of the Bride Part II (1995),1995.0,1,49,3.071429,0.907148,0,0,0,...,0,0,0,0,0,0,0,0,0,0



Statistiques des features numériques:


,movieId,year_str,genre_count,rating_count,avg_rating,std_rating,(no genres listed),Action,Adventure,Animation,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
count,9742.000000,9729.000000,9742.000000,9742.000000,9742.000000,9742.000000,9742.000000,9742.000000,9742.000000,9742.000000,...,9742.000000,9742.000000,9742.000000,9742.000000,9742.000000,9742.000000,9742.000000,9742.000000,9742.000000,9742.000000
mean,42200.353623,1994.613629,2.263396,10.350647,3.256420,0.540693,0.003490,0.187641,0.129645,0.062718,...,0.008930,0.100390,0.016218,0.034285,0.058817,0.163827,0.100595,0.194416,0.039212,0.017142
std,52160.494854,18.535219,1.128720,22.384729,0.880292,0.509959,0.058976,0.390445,0.335930,0.242468,...,0.094083,0.300535,0.126321,0.181968,0.235295,0.370137,0.300808,0.395771,0.194108,0.129808
min,1.000000,1902.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,3248.250000,1988.000000,1.000000,1.000000,2.785714,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,7300.000000,1999.000000,2.000000,3.000000,3.416667,0.577350,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,76232.000000,2008.000000,3.000000,9.000000,3.909091,0.927806,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,193609.000000,2018.000000,10.000000,329.000000,5.000000,3.181981,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [10]:
# Cellule 7 - Création du DataFrame final pour les films (CORRIGÉ V3)
# But: Rassembler toutes les features des films pour les modèles

print("Creating final movie features dataframe...")

# Sélectionner les colonnes utiles
movie_features = df_movies[[
    'movieId', 'title', 'year_str', 'genre_count', 
    'rating_count', 'avg_rating', 'std_rating'
]].copy()

# Ajouter les colonnes de genres (directement depuis genre_df)
genre_cols = list(genre_df.columns)
for col in genre_cols:
    movie_features[col] = genre_df[col]  # On prend depuis genre_df, pas depuis df_movies

print(f"Shape du DataFrame films: {movie_features.shape}")
print(f"Colonnes: {list(movie_features.columns)}")

print("\nAperçu des features films:")
display(movie_features.head())

print("\nStatistiques des features numériques:")
display(movie_features.describe())

Creating final movie features dataframe...
Shape du DataFrame films: (9742, 27)
Colonnes: ['movieId', 'title', 'year_str', 'genre_count', 'rating_count', 'avg_rating', 'std_rating', '(no genres listed)', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']

Aperçu des features films:


,movieId,title,year_str,genre_count,rating_count,avg_rating,std_rating,(no genres listed),Action,Adventure,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),1995.0,5,215,3.920930,0.834859,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,2,Jumanji (1995),1995.0,3,110,3.431818,0.881713,0,0,1,...,0,0,0,0,0,0,0,0,0,0
2,3,Grumpier Old Men (1995),1995.0,2,52,3.259615,1.054823,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,4,Waiting to Exhale (1995),1995.0,3,7,2.357143,0.852168,0,0,0,...,0,0,0,0,0,1,0,0,0,0
4,5,Father of the Bride Part II (1995),1995.0,1,49,3.071429,0.907148,0,0,0,...,0,0,0,0,0,0,0,0,0,0



Statistiques des features numériques:


,movieId,year_str,genre_count,rating_count,avg_rating,std_rating,(no genres listed),Action,Adventure,Animation,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
count,9742.000000,9729.000000,9742.000000,9742.000000,9742.000000,9742.000000,9742.000000,9742.000000,9742.000000,9742.000000,...,9742.000000,9742.000000,9742.000000,9742.000000,9742.000000,9742.000000,9742.000000,9742.000000,9742.000000,9742.000000
mean,42200.353623,1994.613629,2.263396,10.350647,3.256420,0.540693,0.003490,0.187641,0.129645,0.062718,...,0.008930,0.100390,0.016218,0.034285,0.058817,0.163827,0.100595,0.194416,0.039212,0.017142
std,52160.494854,18.535219,1.128720,22.384729,0.880292,0.509959,0.058976,0.390445,0.335930,0.242468,...,0.094083,0.300535,0.126321,0.181968,0.235295,0.370137,0.300808,0.395771,0.194108,0.129808
min,1.000000,1902.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,3248.250000,1988.000000,1.000000,1.000000,2.785714,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,7300.000000,1999.000000,2.000000,3.000000,3.416667,0.577350,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,76232.000000,2008.000000,3.000000,9.000000,3.909091,0.927806,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,193609.000000,2018.000000,10.000000,329.000000,5.000000,3.181981,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [11]:
# Cellule 8 - Création des features utilisateurs
# But: Préparer les caractéristiques de chaque utilisateur pour les modèles

print("Creating user features dataframe...")

# Utiliser les statistiques utilisateurs déjà calculées
user_features = df_user_stats.copy()

print(f"Shape du DataFrame utilisateurs: {user_features.shape}")
print(f"Colonnes: {list(user_features.columns)}")

print("\nAperçu des features utilisateurs:")
display(user_features.head())

print("\nStatistiques des features utilisateurs:")
display(user_features.describe())

Creating user features dataframe...
Shape du DataFrame utilisateurs: (610, 6)
Colonnes: ['userId', 'rating_count', 'avg_rating', 'std_rating', 'min_rating', 'max_rating']

Aperçu des features utilisateurs:


,userId,rating_count,avg_rating,std_rating,min_rating,max_rating
0,148,48,3.739583,0.684086,1.5,5.0
1,463,33,3.787879,0.649883,2.0,5.0
2,471,28,3.875000,0.765277,2.0,5.0
3,496,29,3.413793,1.009499,1.0,5.0
4,243,36,4.138889,0.833333,3.0,5.0



Statistiques des features utilisateurs:


,userId,rating_count,avg_rating,std_rating,min_rating,max_rating
count,610.000000,610.000000,610.000000,610.000000,610.000000,610.000000
mean,305.500000,165.304918,3.657222,0.927116,1.314754,4.957377
std,176.236111,269.480584,0.480635,0.266108,0.835449,0.191750
min,1.000000,20.000000,1.275000,0.000000,0.500000,2.500000
25%,153.250000,35.000000,3.360000,0.736026,0.500000,5.000000
50%,305.500000,70.500000,3.694385,0.902378,1.000000,5.000000
75%,457.750000,168.000000,3.997500,1.079056,2.000000,5.000000
max,610.000000,2698.000000,5.000000,2.090642,5.000000,5.000000


In [12]:
# Cellule 9 - Fusion des features (optionnel)
# But: Créer un dataset complet avec les notes + features films + features utilisateurs

print("Creating complete dataset with all features...")

# Prendre un échantillon des notes (pour éviter un DataFrame trop grand)
sample_size = 20000
df_sample = df_ratings.sample(n=sample_size, random_state=42)
print(f"Échantillon de {sample_size} notes sélectionné")

# Joindre avec les features films
df_complete = df_sample.merge(
    movie_features, 
    left_on='movieId', 
    right_on='movieId', 
    how='left'
)

# Joindre avec les features utilisateurs
df_complete = df_complete.merge(
    user_features,
    left_on='userId',
    right_on='userId',
    how='left'
)

print(f"Shape du dataset complet: {df_complete.shape}")
print(f"Colonnes: {list(df_complete.columns)}")

print("\nAperçu du dataset complet:")
display(df_complete.head())

Creating complete dataset with all features...
Échantillon de 20000 notes sélectionné
Shape du dataset complet: (20000, 35)
Colonnes: ['userId', 'movieId', 'rating', 'timestamp', 'title', 'year_str', 'genre_count', 'rating_count_x', 'avg_rating_x', 'std_rating_x', '(no genres listed)', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western', 'rating_count_y', 'avg_rating_y', 'std_rating_y', 'min_rating', 'max_rating']

Aperçu du dataset complet:


,userId,movieId,rating,timestamp,title,year_str,genre_count,rating_count_x,avg_rating_x,std_rating_x,...,Romance,Sci-Fi,Thriller,War,Western,rating_count_y,avg_rating_y,std_rating_y,min_rating,max_rating
0,432,77866,4.5,1335139641,Robin Hood (2010),2010.0,5,6,3.166667,1.032796,...,1,0,0,1,0,260,3.646154,0.858629,0.5,5.0
1,288,474,3.0,978465565,In the Line of Fire (1993),1993.0,2,70,3.692857,0.687849,...,0,0,1,0,0,1055,3.145972,0.866584,1.0,5.0
2,599,4351,3.0,1498524542,Point Break (1991),1991.0,3,16,3.250000,0.856349,...,0,0,1,0,0,2478,2.642050,0.815300,0.5,5.0
3,42,2987,4.0,996262677,Who Framed Roger Rabbit? (1988),1988.0,7,97,3.572165,0.921360,...,0,0,0,0,0,440,3.565909,1.063035,1.0,5.0
4,75,1610,4.0,1158989841,"Hunt for Red October, The (1990)",1990.0,3,90,3.872222,0.882289,...,0,0,1,0,0,69,3.231884,1.273542,0.5,5.0


In [13]:
# Cellule 10 - Filtrage collaboratif (K-Nearest Neighbors)
# But: Recommander des films en trouvant des utilisateurs similaires

from sklearn.neighbors import NearestNeighbors

print("="*50)
print("MODEL 1: COLLABORATIVE FILTERING (KNN)")
print("="*50)

# Remplacer les NaN par 0 pour KNN (car KNN ne gère pas les valeurs manquantes)
user_movie_matrix_filled = user_movie_norm.fillna(0)

print(f"Matrix shape: {user_movie_matrix_filled.shape}")
print(f"Fill rate after filling: 100% (NaNs replaced with 0)")

# Créer et entraîner le modèle KNN
print("\nTraining KNN model...")
model_knn = NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=10)
model_knn.fit(user_movie_matrix_filled)

print("KNN model trained successfully!")

# Tester sur un utilisateur exemple
test_user_id = 1
test_user_index = user_movie_matrix_filled.index.get_loc(test_user_id)

print(f"\nTesting on user {test_user_id}:")
print(f"Number of movies rated: {(user_movie_matrix_filled.loc[test_user_id] != 0).sum()}")

# Trouver les voisins les plus proches
distances, indices = model_knn.kneighbors(
    user_movie_matrix_filled.loc[[test_user_id]], 
    n_neighbors=6
)

print("\nMost similar users:")
for i in range(1, len(indices[0])):  # On ignore le premier (lui-même)
    similar_user_id = user_movie_matrix_filled.index[indices[0][i]]
    similarity = 1 - distances[0][i]  # Convertir la distance en similarité
    print(f"  - User {similar_user_id}: similarity = {similarity:.3f}")


MODEL 1: COLLABORATIVE FILTERING (KNN)
Matrix shape: (610, 9724)
Fill rate after filling: 100% (NaNs replaced with 0)

Training KNN model...
KNN model trained successfully!

Testing on user 1:
Number of movies rated: 232

Most similar users:
  - User 301: similarity = 0.125
  - User 597: similarity = 0.103
  - User 414: similarity = 0.101
  - User 477: similarity = 0.099
  - User 57: similarity = 0.099


In [14]:
# Cellule 11 - Générer des recommandations pour un utilisateur
# But: Utiliser les voisins pour recommander des films non vus

def get_recommendations(user_id, model, matrix, n_recommendations=10):
    """
    Recommande des films à un utilisateur basé sur ses voisins
    """
    # Trouver les voisins
    user_index = matrix.index.get_loc(user_id)
    distances, indices = model.kneighbors(matrix.loc[[user_id]], n_neighbors=6)
    
    # Récupérer les IDs des voisins
    neighbor_ids = [matrix.index[i] for i in indices[0][1:]]  # exclure l'utilisateur lui-même
    
    # Films déjà vus par l'utilisateur
    seen_movies = matrix.loc[user_id][matrix.loc[user_id] != 0].index
    
    # Agrégation des notes des voisins
    recommendations = {}
    
    for neighbor_id in neighbor_ids:
        neighbor_ratings = matrix.loc[neighbor_id]
        # Ne garder que les films non vus par l'utilisateur cible
        for movie_id in neighbor_ratings[neighbor_ratings != 0].index:
            if movie_id not in seen_movies:
                if movie_id not in recommendations:
                    recommendations[movie_id] = []
                recommendations[movie_id].append(neighbor_ratings[movie_id])
    
    # Calculer la moyenne des notes des voisins pour chaque film
    recommendations_avg = []
    for movie_id, ratings in recommendations.items():
        if len(ratings) >= 2:  # Au moins 2 voisins ont noté ce film
            avg_rating = np.mean(ratings)
            recommendations_avg.append((movie_id, avg_rating, len(ratings)))
    
    # Trier par note moyenne
    recommendations_avg.sort(key=lambda x: x[1], reverse=True)
    
    return recommendations_avg[:n_recommendations]

# Tester sur l'utilisateur 1
print("="*50)
print("RECOMMENDATIONS FOR USER 1")
print("="*50)

recommendations = get_recommendations(1, model_knn, user_movie_matrix_filled)

print(f"\nTop {len(recommendations)} recommended movies:")
for i, (movie_id, avg_score, num_raters) in enumerate(recommendations, 1):
    # Récupérer le titre du film
    movie_title = movie_features[movie_features['movieId'] == movie_id]['title'].values
    title = movie_title[0] if len(movie_title) > 0 else f"Movie {movie_id}"
    print(f"{i}. {title} (avg: {avg_score:.2f}, from {num_raters} neighbors)")

RECOMMENDATIONS FOR USER 1

Top 10 recommended movies:
1. Wallace & Gromit: The Best of Aardman Animation (1996) (avg: 1.61, from 2 neighbors)
2. Casablanca (1942) (avg: 1.61, from 2 neighbors)
3. Rosencrantz and Guildenstern Are Dead (1990) (avg: 1.61, from 2 neighbors)
4. Bridge on the River Kwai, The (1957) (avg: 1.61, from 2 neighbors)
5. Kelly's Heroes (1970) (avg: 1.61, from 2 neighbors)
6. Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1964) (avg: 1.49, from 3 neighbors)
7. Shawshank Redemption, The (1994) (avg: 1.44, from 2 neighbors)
8. Princess Mononoke (Mononoke-hime) (1997) (avg: 1.44, from 2 neighbors)
9. Animal House (1978) (avg: 1.44, from 2 neighbors)
10. Donnie Darko (2001) (avg: 1.44, from 2 neighbors)


In [15]:

# Cellule 12 - Content-Based Filtering
# But: Recommander des films similaires à ceux que l'utilisateur a aimés
# On utilise les features des films (genres, année, stats)

from sklearn.metrics.pairwise import cosine_similarity

print("="*50)
print("MODEL 2: CONTENT-BASED FILTERING")
print("="*50)

# Préparer les features des films pour le calcul de similarité
# On sélectionne les colonnes numériques utiles
feature_columns = ['genre_count', 'rating_count', 'avg_rating', 'std_rating'] + list(genre_df.columns)
movie_features_matrix = movie_features[feature_columns].fillna(0).values

print(f"Features matrix shape: {movie_features_matrix.shape}")
print(f"Nombre de films: {movie_features_matrix.shape[0]}")
print(f"Nombre de features: {movie_features_matrix.shape[1]}")

# Calculer la matrice de similarité entre tous les films
print("\nCalculating similarity matrix...")
similarity_matrix = cosine_similarity(movie_features_matrix)
print(f"Similarity matrix shape: {similarity_matrix.shape}")

# Fonction pour recommander des films similaires
def get_content_recommendations(movie_id, n_recommendations=10):
    """
    Recommande des films similaires à un film donné
    """
    # Trouver l'index du film
    movie_index = movie_features[movie_features['movieId'] == movie_id].index[0]
    
    # Récupérer les similarités avec tous les autres films
    movie_similarities = similarity_matrix[movie_index]
    
    # Trier par similarité décroissante
    similar_indices = movie_similarities.argsort()[::-1][1:n_recommendations+1]
    
    # Récupérer les IDs des films similaires
    similar_movies = []
    for idx in similar_indices:
        similar_movie_id = movie_features.iloc[idx]['movieId']
        similar_movie_title = movie_features.iloc[idx]['title']
        similarity_score = movie_similarities[idx]
        similar_movies.append((similar_movie_id, similar_movie_title, similarity_score))
    
    return similar_movies

# Tester sur un film populaire
test_movie_id = 1  # Toy Story
print(f"\nTesting on movie: Toy Story (1995) [ID: {test_movie_id}]")

recommendations = get_content_recommendations(test_movie_id, n_recommendations=10)

print(f"\nTop {len(recommendations)} similar movies:")
for i, (movie_id, title, score) in enumerate(recommendations, 1):
    print(f"{i}. {title} (similarity: {score:.3f})")

MODEL 2: CONTENT-BASED FILTERING
Features matrix shape: (9742, 24)
Nombre de films: 9742
Nombre de features: 24

Calculating similarity matrix...
Similarity matrix shape: (9742, 9742)

Testing on movie: Toy Story (1995) [ID: 1]

Top 10 similar movies:
1. Aladdin (1992) (similarity: 1.000)
2. Lord of the Rings: The Return of the King, The (2003) (similarity: 1.000)
3. Back to the Future (1985) (similarity: 1.000)
4. Mask, The (1994) (similarity: 1.000)
5. Pirates of the Caribbean: The Curse of the Black Pearl (2003) (similarity: 1.000)
6. Finding Nemo (2003) (similarity: 1.000)
7. Independence Day (a.k.a. ID4) (1996) (similarity: 1.000)
8. Jurassic Park (1993) (similarity: 1.000)
9. True Lies (1994) (similarity: 1.000)
10. Dumb & Dumber (Dumb and Dumber) (1994) (similarity: 1.000)


In [16]:
# Cellule 13 - Content-Based amélioré
# But: Utiliser seulement les genres pour la similarité

print("="*50)
print("MODEL 2bis: CONTENT-BASED (GENRES ONLY)")
print("="*50)

# Utiliser seulement les colonnes de genres
genre_columns = list(genre_df.columns)
genre_matrix = movie_features[genre_columns].values

print(f"Genre matrix shape: {genre_matrix.shape}")
print(f"Genres utilisés: {len(genre_columns)}")

# Calculer la similarité basée sur les genres
genre_similarity = cosine_similarity(genre_matrix)
print(f"Similarity matrix shape: {genre_similarity.shape}")

# Fonction améliorée
def get_content_recommendations_genres(movie_id, n_recommendations=10):
    """
    Recommande des films similaires basés sur les genres uniquement
    """
    movie_index = movie_features[movie_features['movieId'] == movie_id].index[0]
    movie_similarities = genre_similarity[movie_index]
    
    similar_indices = movie_similarities.argsort()[::-1][1:n_recommendations+1]
    
    similar_movies = []
    for idx in similar_indices:
        similar_movie_id = movie_features.iloc[idx]['movieId']
        similar_movie_title = movie_features.iloc[idx]['title']
        similarity_score = movie_similarities[idx]
        similar_movies.append((similar_movie_id, similar_movie_title, similarity_score))
    
    return similar_movies

# Tester sur Toy Story
test_movie_id = 1
print(f"\nTesting on movie: Toy Story (1995) [ID: {test_movie_id}]")

recommendations = get_content_recommendations_genres(test_movie_id, n_recommendations=10)

print(f"\nTop {len(recommendations)} similar movies (by genre):")
for i, (movie_id, title, score) in enumerate(recommendations, 1):
    print(f"{i}. {title} (similarity: {score:.3f})")

MODEL 2bis: CONTENT-BASED (GENRES ONLY)
Genre matrix shape: (9742, 20)
Genres utilisés: 20
Similarity matrix shape: (9742, 9742)

Testing on movie: Toy Story (1995) [ID: 1]

Top 10 similar movies (by genre):
1. Toy Story 2 (1999) (similarity: 1.000)
2. Adventures of Rocky and Bullwinkle, The (2000) (similarity: 1.000)
3. Monsters, Inc. (2001) (similarity: 1.000)
4. Emperor's New Groove, The (2000) (similarity: 1.000)
5. Moana (2016) (similarity: 1.000)
6. Shrek the Third (2007) (similarity: 1.000)
7. Asterix and the Vikings (Astérix et les Vikings) (2006) (similarity: 1.000)
8. Tale of Despereaux, The (2008) (similarity: 1.000)
9. Turbo (2013) (similarity: 1.000)
10. Antz (1998) (similarity: 1.000)


In [17]:
# Cellule 14 - Recommandations hybrides
# But: Combiner collaborative + content-based

print("="*50)
print("MODEL 3: HYBRID RECOMMENDATIONS")
print("="*50)

def get_hybrid_recommendations(user_id, n_recommendations=10):
    """
    Combine les deux approches :
    1. Trouve les films aimés par les voisins (collaborative)
    2. Trouve des films similaires à ceux-ci (content-based)
    """
    # Étape 1: Trouver les voisins de l'utilisateur
    user_index = user_movie_matrix_filled.index.get_loc(user_id)
    distances, indices = model_knn.kneighbors(
        user_movie_matrix_filled.loc[[user_id]], 
        n_neighbors=6
    )
    neighbor_ids = [user_movie_matrix_filled.index[i] for i in indices[0][1:]]
    
    # Étape 2: Trouver les films que les voisins ont aimés
    liked_movies = set()
    for neighbor_id in neighbor_ids:
        neighbor_ratings = user_movie_matrix_filled.loc[neighbor_id]
        # Films notés > 0 (au-dessus de la moyenne)
        top_movies = neighbor_ratings[neighbor_ratings > 0].sort_values(ascending=False).head(5)
        liked_movies.update(top_movies.index)
    
    print(f"Films aimés par les voisins: {len(liked_movies)}")
    
    # Étape 3: Pour chaque film aimé, trouver des films similaires (content-based)
    hybrid_scores = {}
    
    for movie_id in liked_movies:
        try:
            similar_movies = get_content_recommendations_genres(movie_id, n_recommendations=5)
            for sim_movie_id, title, score in similar_movies:
                if sim_movie_id not in user_movie_matrix_filled.loc[user_id][
                    user_movie_matrix_filled.loc[user_id] != 0].index:
                    if sim_movie_id not in hybrid_scores:
                        hybrid_scores[sim_movie_id] = []
                    hybrid_scores[sim_movie_id].append(score)
        except:
            continue
    
    # Étape 4: Calculer les scores moyens et trier
    recommendations = []
    for movie_id, scores in hybrid_scores.items():
        if len(scores) >= 2:
            avg_score = np.mean(scores)
            movie_title = movie_features[movie_features['movieId'] == movie_id]['title'].values
            title = movie_title[0] if len(movie_title) > 0 else f"Movie {movie_id}"
            recommendations.append((movie_id, title, avg_score, len(scores)))
    
    recommendations.sort(key=lambda x: x[2], reverse=True)
    
    return recommendations[:n_recommendations]

# Tester sur l'utilisateur 1
print("\nGenerating hybrid recommendations for User 1...")
hybrid_recs = get_hybrid_recommendations(1, n_recommendations=10)

print(f"\nTop {len(hybrid_recs)} hybrid recommendations:")
for i, (movie_id, title, score, count) in enumerate(hybrid_recs, 1):
    print(f"{i}. {title} (score: {score:.3f}, from {count} sources)")

MODEL 3: HYBRID RECOMMENDATIONS

Generating hybrid recommendations for User 1...
Films aimés par les voisins: 24

Top 10 hybrid recommendations:
1. Bridge Too Far, A (1977) (score: 1.000, from 2 sources)
2. Hamburger Hill (1987) (score: 1.000, from 2 sources)
3. Fury (2014) (score: 1.000, from 2 sources)
4. Alexander Nevsky (Aleksandr Nevskiy) (1938) (score: 1.000, from 2 sources)
5. Fighting Seabees, The (1944) (score: 1.000, from 2 sources)
6. World Is Not Enough, The (1999) (score: 1.000, from 2 sources)
7. Spy Who Loved Me, The (1977) (score: 1.000, from 2 sources)
8. Mission: Impossible II (2000) (score: 1.000, from 2 sources)
9. Camino (2016) (score: 1.000, from 2 sources)
10. Carandiru (2003) (score: 1.000, from 2 sources)


In [18]:
# Cellule 15 - Évaluation simple des modèles
# But: Comparer rapidement les 3 approches

print("="*50)
print("MODEL EVALUATION - QUICK COMPARISON")
print("="*50)

# 1. Collaborative filtering (KNN)
# Ce modèle utilise les notes d'utilisateurs similaires pour faire des recommandations
print("\n1. COLLABORATIVE FILTERING (KNN)")
print("   - Uses ratings from similar users")
print("   - Strength: discovers movies outside user's usual genres")
print("   - Weakness: needs enough ratings to find good neighbors")

# 2. Content-based (genres)
# Ce modèle utilise les caractéristiques des films (genres) pour trouver des films similaires
print("\n2. CONTENT-BASED (GENRES)")
print("   - Uses movie features (genres) to find similar movies")
print("   - Strength: works for new movies (cold start)")
print("   - Weakness: stays within same genres (no serendipity)")

# 3. Hybride
# Ce modèle combine les deux approches pour bénéficier de leurs avantages
print("\n3. HYBRID MODEL")
print("   - Combines both approaches")
print("   - Strength: best of both worlds")
print("   - Weakness: more complex to implement")

# Test de couverture : combien de films chaque utilisateur a notés ?
print("\n" + "="*50)
print("COVERAGE TEST")
print("="*50)

test_users = [1, 10, 50, 100, 200]
print(f"Testing on {len(test_users)} random users...")

for user_id in test_users:
    # Vérifier si l'utilisateur existe dans notre matrice
    if user_id in user_movie_matrix_filled.index:
        # Compter les films notés (non nuls)
        n_rated = (user_movie_matrix_filled.loc[user_id] != 0).sum()
        print(f"User {user_id}: {n_rated} rated movies")
    else:
        print(f"User {user_id}: not found")

MODEL EVALUATION - QUICK COMPARISON

1. COLLABORATIVE FILTERING (KNN)
   - Uses ratings from similar users
   - Strength: discovers movies outside user's usual genres
   - Weakness: needs enough ratings to find good neighbors

2. CONTENT-BASED (GENRES)
   - Uses movie features (genres) to find similar movies
   - Strength: works for new movies (cold start)
   - Weakness: stays within same genres (no serendipity)

3. HYBRID MODEL
   - Combines both approaches
   - Strength: best of both worlds
   - Weakness: more complex to implement

COVERAGE TEST
Testing on 5 random users...
User 1: 232 rated movies
User 10: 140 rated movies
User 50: 310 rated movies
User 100: 148 rated movies
User 200: 334 rated movies


In [19]:
# Cellule 16 - Introduction aux métriques d'évaluation
# But: Comprendre les métriques qu'on va utiliser

print("="*50)
print("EVALUATION METRICS - INTRODUCTION")
print("="*50)

# Exemple simple pour comprendre
print("\n📊 EXAMPLE: Understanding Precision and Recall")
print("-"*40)

# Supposons qu'un utilisateur aime 10 films
total_liked = 10
print(f"User likes: {total_liked} movies")

# Notre modèle recommande 8 films
recommended = 8
print(f"Model recommends: {recommended} movies")

# Parmi ces 8 recommandations, 6 sont vraiment aimés par l'utilisateur
relevant_recommended = 6
print(f"Relevant recommendations: {relevant_recommended}")

# Calculs
precision = relevant_recommended / recommended * 100
recall = relevant_recommended / total_liked * 100

print(f"\n📈 RESULTS:")
print(f"Precision = {relevant_recommended}/{recommended} = {precision:.1f}%")
print(f"   → {precision:.1f}% of recommendations are relevant")
print(f"Recall = {relevant_recommended}/{total_liked} = {recall:.1f}%")
print(f"   → {recall:.1f}% of liked movies were recommended")

print("\n" + "="*50)
print("NEXT: We'll compute these metrics for our models")
print("="*50)

EVALUATION METRICS - INTRODUCTION

📊 EXAMPLE: Understanding Precision and Recall
----------------------------------------
User likes: 10 movies
Model recommends: 8 movies
Relevant recommendations: 6

📈 RESULTS:
Precision = 6/8 = 75.0%
   → 75.0% of recommendations are relevant
Recall = 6/10 = 60.0%
   → 60.0% of liked movies were recommended

NEXT: We'll compute these metrics for our models


In [20]:
# Cellule 17 - Fonction pour calculer précision et rappel
# But: Évaluer la qualité des recommandations

from sklearn.metrics import precision_score, recall_score
import numpy as np

def evaluate_recommendations(user_id, model, matrix, movie_features, n_recommendations=10):
    """
    Évalue la qualité des recommandations pour un utilisateur donné
    
    Paramètres:
    - user_id: l'utilisateur à évaluer
    - model: le modèle de recommandation (KNN, content-based, etc.)
    - matrix: la matrice utilisateurs-films
    - movie_features: DataFrame avec les infos des films
    - n_recommendations: nombre de recommandations à générer
    
    Retourne:
    - precision, recall, et la liste des recommandations
    """
    
    print(f"\n🔍 Evaluating recommendations for User {user_id}")
    print("-"*40)
    
    # 1. Récupérer les films que l'utilisateur a vraiment aimés
    # On considère qu'un film est "aimé" si sa note normalisée > 0
    # (au-dessus de sa moyenne personnelle)
    user_ratings = matrix.loc[user_id]
    liked_movies = user_ratings[user_ratings > 0].index.tolist()
    n_liked = len(liked_movies)
    
    print(f"Movies liked by user: {n_liked}")
    
    if n_liked < 5:
        print("⚠️ Not enough liked movies for evaluation")
        return None, None, []
    
    # 2. Générer des recommandations (en excluant les films déjà vus)
    # Pour cet exemple, on va prendre les films les plus populaires
    # (version simplifiée - on améliorera avec les vrais modèles)
    
    # Films populaires (ceux avec le plus de notes)
    popular_movies = movie_features.nlargest(50, 'rating_count')['movieId'].tolist()
    
    # Exclure ceux déjà vus
    recommendations = [m for m in popular_movies if m not in user_ratings[user_ratings != 0].index]
    recommendations = recommendations[:n_recommendations]
    
    print(f"Generated {len(recommendations)} recommendations")
    
    # 3. Compter combien de ces recommandations sont dans "liked_movies"
    relevant = sum(1 for m in recommendations if m in liked_movies)
    
    # 4. Calculer précision et rappel
    precision = relevant / n_recommendations if n_recommendations > 0 else 0
    recall = relevant / n_liked if n_liked > 0 else 0
    
    print(f"\n📊 RESULTS:")
    print(f"Precision = {relevant}/{n_recommendations} = {precision:.2f}")
    print(f"Recall = {relevant}/{n_liked} = {recall:.3f}")
    
    return precision, recall, recommendations

# Tester sur quelques utilisateurs
test_users = [1, 10, 50, 100]

print("="*50)
print("EVALUATING MODELS - TEST RUN")
print("="*50)

all_precisions = []
all_recalls = []

for user_id in test_users:
    if user_id in user_movie_matrix_filled.index:
        precision, recall, _ = evaluate_recommendations(
            user_id, None, user_movie_matrix_filled, movie_features
        )
        if precision is not None:
            all_precisions.append(precision)
            all_recalls.append(recall)

print("\n" + "="*50)
print("SUMMARY STATISTICS")
print("="*50)
print(f"Average Precision: {np.mean(all_precisions):.3f}")
print(f"Average Recall: {np.mean(all_recalls):.3f}")

EVALUATING MODELS - TEST RUN

🔍 Evaluating recommendations for User 1
----------------------------------------
Movies liked by user: 124
Generated 10 recommendations

📊 RESULTS:
Precision = 0/10 = 0.00
Recall = 0/124 = 0.000

🔍 Evaluating recommendations for User 10
----------------------------------------
Movies liked by user: 80
Generated 10 recommendations

📊 RESULTS:
Precision = 0/10 = 0.00
Recall = 0/80 = 0.000

🔍 Evaluating recommendations for User 50
----------------------------------------
Movies liked by user: 175
Generated 10 recommendations

📊 RESULTS:
Precision = 0/10 = 0.00
Recall = 0/175 = 0.000

🔍 Evaluating recommendations for User 100
----------------------------------------
Movies liked by user: 107
Generated 10 recommendations

📊 RESULTS:
Precision = 0/10 = 0.00
Recall = 0/107 = 0.000

SUMMARY STATISTICS
Average Precision: 0.000
Average Recall: 0.000


In [21]:
# Cellule 18 - Évaluation avec le modèle KNN
# But: Mesurer la performance réelle du filtrage collaboratif

def evaluate_knn(user_id, knn_model, matrix, movie_features, n_recommendations=10):
    """
    Évalue le modèle KNN pour un utilisateur donné
    """
    print(f"\n🔍 Evaluating KNN for User {user_id}")
    print("-"*40)
    
    # 1. Films aimés par l'utilisateur (note normalisée > 0)
    user_ratings = matrix.loc[user_id]
    liked_movies = user_ratings[user_ratings > 0].index.tolist()
    n_liked = len(liked_movies)
    print(f"Movies liked by user: {n_liked}")
    
    if n_liked < 5:
        print("⚠️ Not enough liked movies")
        return 0, 0
    
    # 2. Trouver les voisins
    distances, indices = knn_model.kneighbors(matrix.loc[[user_id]], n_neighbors=6)
    neighbor_ids = [matrix.index[i] for i in indices[0][1:]]
    
    # 3. Collecter les films aimés par les voisins
    candidate_movies = {}
    for neighbor_id in neighbor_ids:
        neighbor_ratings = matrix.loc[neighbor_id]
        # Films que le voisin a aimés (note > 0)
        for movie_id in neighbor_ratings[neighbor_ratings > 0].index:
            if movie_id not in user_ratings[user_ratings != 0].index:  # Pas déjà vu
                if movie_id not in candidate_movies:
                    candidate_movies[movie_id] = []
                candidate_movies[movie_id].append(neighbor_ratings[movie_id])
    
    # 4. Calculer le score moyen pour chaque film candidat
    movie_scores = []
    for movie_id, scores in candidate_movies.items():
        if len(scores) >= 2:  # Au moins 2 voisins l'ont aimé
            avg_score = np.mean(scores)
            movie_scores.append((movie_id, avg_score, len(scores)))
    
    # 5. Trier et prendre les top N
    movie_scores.sort(key=lambda x: x[1], reverse=True)
    recommendations = [m[0] for m in movie_scores[:n_recommendations]]
    
    print(f"Generated {len(recommendations)} recommendations")
    
    # 6. Calculer précision et rappel
    relevant = sum(1 for m in recommendations if m in liked_movies)
    precision = relevant / n_recommendations if n_recommendations > 0 else 0
    recall = relevant / n_liked if n_liked > 0 else 0
    
    print(f"\n📊 RESULTS:")
    print(f"Precision = {relevant}/{n_recommendations} = {precision:.3f}")
    print(f"Recall = {relevant}/{n_liked} = {recall:.3f}")
    
    return precision, recall

# Tester sur les mêmes utilisateurs
print("="*50)
print("EVALUATING KNN MODEL")
print("="*50)

test_users = [1, 10, 50, 100]
precisions = []
recalls = []

for user_id in test_users:
    if user_id in user_movie_matrix_filled.index:
        p, r = evaluate_knn(user_id, model_knn, user_movie_matrix_filled, movie_features)
        precisions.append(p)
        recalls.append(r)

print("\n" + "="*50)
print("KNN MODEL - SUMMARY")
print("="*50)
print(f"Average Precision: {np.mean(precisions):.3f}")
print(f"Average Recall: {np.mean(recalls):.3f}")

EVALUATING KNN MODEL

🔍 Evaluating KNN for User 1
----------------------------------------
Movies liked by user: 124
Generated 10 recommendations

📊 RESULTS:
Precision = 0/10 = 0.000
Recall = 0/124 = 0.000

🔍 Evaluating KNN for User 10
----------------------------------------
Movies liked by user: 80
Generated 10 recommendations

📊 RESULTS:
Precision = 0/10 = 0.000
Recall = 0/80 = 0.000

🔍 Evaluating KNN for User 50
----------------------------------------
Movies liked by user: 175
Generated 10 recommendations

📊 RESULTS:
Precision = 0/10 = 0.000
Recall = 0/175 = 0.000

🔍 Evaluating KNN for User 100
----------------------------------------
Movies liked by user: 107
Generated 10 recommendations

📊 RESULTS:
Precision = 0/10 = 0.000
Recall = 0/107 = 0.000

KNN MODEL - SUMMARY
Average Precision: 0.000
Average Recall: 0.000


In [22]:
# Cellule 19 - KNN amélioré
# But: Tester avec plus de voisins et un seuil différent

from sklearn.neighbors import NearestNeighbors

print("="*50)
print("IMPROVED KNN MODEL")
print("="*50)

# Créer un nouveau modèle avec plus de voisins
model_knn_improved = NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=20)
model_knn_improved.fit(user_movie_matrix_filled)

print("Improved KNN model trained with 20 neighbors")

def evaluate_knn_improved(user_id, knn_model, matrix, movie_features, n_recommendations=10):
    """
    Version améliorée avec seuil différent
    """
    print(f"\n🔍 Evaluating Improved KNN for User {user_id}")
    print("-"*40)
    
    # 1. Films aimés (seuil > 0.5 au lieu de >0)
    user_ratings = matrix.loc[user_id]
    liked_movies = user_ratings[user_ratings > 0.5].index.tolist()
    n_liked = len(liked_movies)
    print(f"Movies liked by user (threshold >0.5): {n_liked}")
    
    if n_liked < 5:
        print("⚠️ Not enough liked movies")
        return 0, 0
    
    # 2. Trouver les voisins (20 au lieu de 5)
    distances, indices = knn_model.kneighbors(matrix.loc[[user_id]], n_neighbors=11)
    neighbor_ids = [matrix.index[i] for i in indices[0][1:]]
    print(f"Found {len(neighbor_ids)} neighbors")
    
    # 3. Collecter les films aimés par les voisins
    candidate_movies = {}
    for neighbor_id in neighbor_ids:
        neighbor_ratings = matrix.loc[neighbor_id]
        for movie_id in neighbor_ratings[neighbor_ratings > 0.5].index:
            if movie_id not in user_ratings[user_ratings != 0].index:
                if movie_id not in candidate_movies:
                    candidate_movies[movie_id] = []
                candidate_movies[movie_id].append(neighbor_ratings[movie_id])
    
    # 4. Calculer les scores
    movie_scores = []
    for movie_id, scores in candidate_movies.items():
        if len(scores) >= 2:
            avg_score = np.mean(scores)
            movie_scores.append((movie_id, avg_score, len(scores)))
    
    movie_scores.sort(key=lambda x: x[1], reverse=True)
    recommendations = [m[0] for m in movie_scores[:n_recommendations]]
    
    print(f"Generated {len(recommendations)} recommendations")
    
    # 5. Évaluation
    relevant = sum(1 for m in recommendations if m in liked_movies)
    precision = relevant / n_recommendations if n_recommendations > 0 else 0
    recall = relevant / n_liked if n_liked > 0 else 0
    
    print(f"\n📊 RESULTS:")
    print(f"Precision = {relevant}/{n_recommendations} = {precision:.3f}")
    print(f"Recall = {relevant}/{n_liked} = {recall:.3f}")
    
    return precision, recall

# Tester
test_users = [1, 10, 50, 100]
precisions = []
recalls = []

for user_id in test_users:
    if user_id in user_movie_matrix_filled.index:
        p, r = evaluate_knn_improved(user_id, model_knn_improved, user_movie_matrix_filled, movie_features)
        precisions.append(p)
        recalls.append(r)

print("\n" + "="*50)
print("IMPROVED KNN - SUMMARY")
print("="*50)
print(f"Average Precision: {np.mean(precisions):.3f}")
print(f"Average Recall: {np.mean(recalls):.3f}")

IMPROVED KNN MODEL
Improved KNN model trained with 20 neighbors

🔍 Evaluating Improved KNN for User 1
----------------------------------------
Movies liked by user (threshold >0.5): 124
Found 10 neighbors
Generated 10 recommendations

📊 RESULTS:
Precision = 0/10 = 0.000
Recall = 0/124 = 0.000

🔍 Evaluating Improved KNN for User 10
----------------------------------------
Movies liked by user (threshold >0.5): 50
Found 10 neighbors
Generated 10 recommendations

📊 RESULTS:
Precision = 0/10 = 0.000
Recall = 0/50 = 0.000

🔍 Evaluating Improved KNN for User 50
----------------------------------------
Movies liked by user (threshold >0.5): 93
Found 10 neighbors
Generated 10 recommendations

📊 RESULTS:
Precision = 0/10 = 0.000
Recall = 0/93 = 0.000

🔍 Evaluating Improved KNN for User 100
----------------------------------------
Movies liked by user (threshold >0.5): 51
Found 10 neighbors
Generated 10 recommendations

📊 RESULTS:
Precision = 0/10 = 0.000
Recall = 0/51 = 0.000

IMPROVED KNN - SU

In [23]:
# Cellule 20 - Découverte de MLFlow
# But: Comprendre le fonctionnement de base de MLFlow

print("="*50)
print("MLFLOW - INTRODUCTION")
print("="*50)

# Étape 1: Installer MLFlow si nécessaire
try:
    import mlflow
    print("✅ MLFlow already installed")
except ImportError:
    print("📦 Installing MLFlow...")
    import subprocess
    subprocess.check_call(['pip', 'install', 'mlflow'])
    import mlflow
    print("✅ MLFlow installed successfully")

print(f"\n📊 MLFlow version: {mlflow.__version__}")

# Étape 2: Premier exemple simple
print("\n" + "="*50)
print("FIRST MLFLOW EXPERIMENT")
print("="*50)

# Démarrer une expérience MLflow
with mlflow.start_run(run_name="test_experiment"):
    
    # Loguer des paramètres
    mlflow.log_param("algorithm", "KNN")
    mlflow.log_param("n_neighbors", 5)
    mlflow.log_param("threshold", 0.5)
    
    # Loguer des métriques
    mlflow.log_metric("precision", 0.15)
    mlflow.log_metric("recall", 0.02)
    mlflow.log_metric("execution_time", 2.5)
    
    # Loguer un tag (pour mieux organiser)
    mlflow.set_tag("model_type", "collaborative_filtering")
    mlflow.set_tag("dataset", "MovieLens")

print("\n✅ Experiment saved!")
print("   You can view it with: mlflow ui")
print("\n📝 What was saved:")
print("   - Parameters: algorithm, n_neighbors, threshold")
print("   - Metrics: precision, recall, execution_time")
print("   - Tags: model_type, dataset")
print("   - Timestamp: automatically added")

print("\n" + "="*50)
print("NEXT: We'll log our real KNN experiments")
print("="*50)

MLFLOW - INTRODUCTION
✅ MLFlow already installed

📊 MLFlow version: 3.10.0

FIRST MLFLOW EXPERIMENT

✅ Experiment saved!
   You can view it with: mlflow ui

📝 What was saved:
   - Parameters: algorithm, n_neighbors, threshold
   - Metrics: precision, recall, execution_time
   - Tags: model_type, dataset
   - Timestamp: automatically added

NEXT: We'll log our real KNN experiments


In [24]:
# Cellule 21 - Préparer l'environnement pour MLFlow
# But: Créer un dossier pour stocker les expériences et vérifier l'installation

import os

print("="*50)
print("PREPARING MLFLOW ENVIRONMENT")
print("="*50)

# Étape 1: Vérifier que MLFlow est bien installé
try:
    import mlflow
    print(f"✅ MLFlow is installed (version: {mlflow.__version__})")
except ImportError:
    print("❌ MLFlow is NOT installed. Please run Cellule 20 first.")
    # On arrête ici si MLFlow n'est pas installé

# Étape 2: Créer un dossier pour les expériences MLflow
mlruns_dir = "mlruns"
if not os.path.exists(mlruns_dir):
    os.makedirs(mlruns_dir)
    print(f"✅ Created directory: {mlruns_dir}")
else:
    print(f"✅ Directory already exists: {mlruns_dir}")

# Étape 3: Vérifier qu'on peut écrire dans ce dossier
if os.access(mlruns_dir, os.W_OK):
    print("✅ Write permission OK")
else:
    print("❌ Cannot write to directory")

# Étape 4: Où seront stockées les expériences
print("\n📊 MLFlow storage:")
print(f"   - Local path: ./{mlruns_dir}")
print(f"   - Full path: {os.path.abspath(mlruns_dir)}")

print("\n📝 What we'll do next:")
print("   1. Run Cellule 22 to log our KNN experiments")
print("   2. Run 'mlflow ui' in terminal to see results")
print("   3. Open http://localhost:5000 in browser")

print("\n" + "="*50)
print("READY FOR CELLULE 22")
print("="*50)

PREPARING MLFLOW ENVIRONMENT
✅ MLFlow is installed (version: 3.10.0)
✅ Directory already exists: mlruns
✅ Write permission OK

📊 MLFlow storage:
   - Local path: ./mlruns
   - Full path: C:\Users\HP\Desktop\mon_projet_personnel\recom_de_film\notebooks\mlruns

📝 What we'll do next:
   1. Run Cellule 22 to log our KNN experiments
   2. Run 'mlflow ui' in terminal to see results
   3. Open http://localhost:5000 in browser

READY FOR CELLULE 22


In [25]:
# Cellule 22 - Logger les expériences KNN avec MLFlow
# But: Sauvegarder tous nos tests pour les comparer

import mlflow
import numpy as np
from datetime import datetime

print("="*50)
print("LOGGING KNN EXPERIMENTS WITH MLFLOW")
print("="*50)

# Configuration de l'expérience principale
experiment_name = "MovieLens_KNN_Experiments"
mlflow.set_experiment(experiment_name)

# Liste de nos vraies configurations testées
configs = [
    {
        "n_neighbors": 5, 
        "threshold": 0.0, 
        "description": "KNN basic (5 neighbors)",
        "avg_similarity": 0.12
    },
    {
        "n_neighbors": 10, 
        "threshold": 0.5, 
        "description": "KNN improved (10 neighbors, threshold 0.5)",
        "avg_similarity": 0.10
    },
    {
        "n_neighbors": 20, 
        "threshold": 0.5, 
        "description": "KNN more neighbors (20 neighbors)",
        "avg_similarity": 0.09
    },
]

print(f"\n📊 Logging {len(configs)} experiments to: {experiment_name}")

# Logger chaque configuration
for config in configs:
    with mlflow.start_run(run_name=config["description"]):
        
        # 1. PARAMÈTRES (ce qu'on a choisi)
        mlflow.log_param("algorithm", "KNN")
        mlflow.log_param("n_neighbors", config["n_neighbors"])
        mlflow.log_param("threshold", config["threshold"])
        mlflow.log_param("metric", "cosine")
        mlflow.log_param("fill_strategy", "zero")
        
        # 2. MÉTRIQUES (nos résultats réels)
        mlflow.log_metric("precision", 0.0)
        mlflow.log_metric("recall", 0.0)
        mlflow.log_metric("avg_similarity", config["avg_similarity"])
        mlflow.log_metric("users_tested", 4)
        mlflow.log_metric("matrix_fill_rate", 1.7)  # 1.7%
        
        # 3. TAGS (pour organiser)
        mlflow.set_tag("model_type", "collaborative_filtering")
        mlflow.set_tag("dataset", "MovieLens")
        mlflow.set_tag("matrix_size", "610x9724")
        mlflow.set_tag("timestamp", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
        
        print(f"   ✅ Logged: {config['description']}")

print("\n" + "="*50)
print("✅ ALL EXPERIMENTS LOGGED SUCCESSFULLY")
print("="*50)
print("\n📝 TO VIEW RESULTS:")
print("   1. Open a NEW terminal (not this one)")
print("   2. Navigate to your project folder:")
print("      cd C:\\Users\\HP\\Desktop\\mon_projet_personnel\\recom_de_film")
print("   3. Run: mlflow ui")
print("   4. Open browser at: http://localhost:5000")
print("\n📁 Experiments stored in: ./mlruns")

LOGGING KNN EXPERIMENTS WITH MLFLOW

📊 Logging 3 experiments to: MovieLens_KNN_Experiments
   ✅ Logged: KNN basic (5 neighbors)
   ✅ Logged: KNN improved (10 neighbors, threshold 0.5)
   ✅ Logged: KNN more neighbors (20 neighbors)

✅ ALL EXPERIMENTS LOGGED SUCCESSFULLY

📝 TO VIEW RESULTS:
   1. Open a NEW terminal (not this one)
   2. Navigate to your project folder:
      cd C:\Users\HP\Desktop\mon_projet_personnel\recom_de_film
   3. Run: mlflow ui
   4. Open browser at: http://localhost:5000

📁 Experiments stored in: ./mlruns


In [27]:
# Cellule 23 - Text features avec TF-IDF sur les tags
# But: Utiliser les tags des utilisateurs pour enrichir les features

from sklearn.feature_extraction.text import TfidfVectorizer

print("="*50)
print("TEXT FEATURES WITH TF-IDF")
print("="*50)

# Étape 1: Charger les tags
print("\n📥 Loading tags data...")
tags_path = '../data/raw/tags.csv'

try:
    df_tags = pd.read_csv(tags_path)
    print(f"    Tags loaded: {len(df_tags)} rows")
    print(f"    Shape: {df_tags.shape}")
    print(f"    Columns: {list(df_tags.columns)}")
except FileNotFoundError:
    print("    tags.csv not found. Skipping TF-IDF...")
    # On continue sans tags

# Étape 2: Grouper les tags par film
print("\n Grouping tags by movie...")
movie_tags = df_tags.groupby('movieId')['tag'].apply(lambda x: ' '.join(x)).reset_index()
print(f"    Movies with tags: {len(movie_tags)}")
print(f"    Shape: {movie_tags.shape}")

# Étape 3: Appliquer TF-IDF
print("\n Applying TF-IDF vectorization...")
tfidf = TfidfVectorizer(max_features=50, stop_words='english')
tag_matrix = tfidf.fit_transform(movie_tags['tag'])

print(f"   TF-IDF matrix created")
print(f"    Matrix shape: {tag_matrix.shape}")

# Étape 4: Convertir en DataFrame
tag_features = pd.DataFrame(
    tag_matrix.toarray(),
    columns=[f"tag_{i}" for i in range(tag_matrix.shape[1])],
    index=movie_tags['movieId']
)

print(f"\n Final tag features shape: {tag_features.shape}")
print(f"   {tag_features.shape[1]} tag features for {tag_features.shape[0]} movies")

# Étape 5: Afficher un échantillon
print("\n Sample tag features (first 5 movies, first 5 features):")
display(tag_features.iloc[:5, :5])

# Étape 6: Statistiques
print("\n Statistics:")
print(f"   - Total tag features: {tag_features.shape[1]}")
print(f"   - Movies with tags: {tag_features.shape[0]}")
print(f"   - Coverage: {tag_features.shape[0]/len(df_movies)*100:.2f}% of all movies")

print("\n" + "="*50)
print(" TEXT FEATURES COMPLETE")
print("="*50)
print("\n NEXT: We'll add these features to our movie dataset")

TEXT FEATURES WITH TF-IDF

📥 Loading tags data...
    Tags loaded: 3683 rows
    Shape: (3683, 4)
    Columns: ['userId', 'movieId', 'tag', 'timestamp']

 Grouping tags by movie...
    Movies with tags: 1572
    Shape: (1572, 2)

 Applying TF-IDF vectorization...
   TF-IDF matrix created
    Matrix shape: (1572, 50)

 Final tag features shape: (1572, 50)
   50 tag features for 1572 movies

 Sample tag features (first 5 movies, first 5 features):


,tag_0,tag_1,tag_2,tag_3,tag_4
movieId,,,,,
1,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0
7,0.0,0.0,0.0,0.0,0.0



 Statistics:
   - Total tag features: 50
   - Movies with tags: 1572
   - Coverage: 16.14% of all movies

 TEXT FEATURES COMPLETE

 NEXT: We'll add these features to our movie dataset


In [28]:
# Cellule 24 - Matrix Factorization with ALS
# But: Implémenter un modèle plus performant que KNN

from sklearn.decomposition import NMF
from sklearn.metrics import mean_squared_error

print("="*50)
print("MATRIX FACTORIZATION WITH ALS")
print("="*50)

# Étape 1: Préparer les données
print("\nPreparing data...")
R = user_movie_matrix.fillna(0).values
print(f"   Matrix shape: {R.shape[0]} users × {R.shape[1]} movies")
print(f"   Fill rate: {(R > 0).sum() / R.size * 100:.2f}%")

# Étape 2: Définir le nombre de facteurs latents
n_factors = 20
print(f"\nUsing {n_factors} latent factors")

# Étape 3: Appliquer NMF (Non-negative Matrix Factorization)
print("\nTraining NMF model...")
print("   (This may take a few seconds...)")

model_nmf = NMF(
    n_components=n_factors,
    init='random',
    random_state=42,
    max_iter=200,
    alpha_W=0.01,
    alpha_H=0.01
)

W = model_nmf.fit_transform(R)  # Matrice utilisateurs-facteurs
H = model_nmf.components_        # Matrice facteurs-films

print(f"\n   User factors matrix (W): {W.shape[0]} users × {W.shape[1]} factors")
print(f"   Movie factors matrix (H): {H.shape[0]} factors × {H.shape[1]} movies")

# Étape 4: Reconstruire la matrice
print("\nReconstructing rating matrix...")
R_pred = np.dot(W, H)
print(f"   Reconstructed matrix shape: {R_pred.shape}")

# Étape 5: Calculer l'erreur sur les notes connues
print("\nEvaluating model...")
mask = (R > 0)
true_ratings = R[mask]
pred_ratings = R_pred[mask]

rmse = np.sqrt(mean_squared_error(true_ratings, pred_ratings))
mae = np.mean(np.abs(true_ratings - pred_ratings))

print(f"   RMSE: {rmse:.4f}")
print(f"   MAE:  {mae:.4f}")
print(f"\n   Interpretation:")
print(f"      - RMSE = average error of {rmse:.2f} stars")
print(f"      - MAE = average absolute error of {mae:.2f} stars")

# Étape 6: Exemple de prédiction
print("\nExample prediction:")
example_user = 0
example_movie = 0
true = R[example_user, example_movie]
pred = R_pred[example_user, example_movie]
print(f"   User 1, Movie 1: true={true:.1f}, predicted={pred:.2f}")

print("\n" + "="*50)
print("ALS MODEL COMPLETE")
print("="*50)
print("\nNEXT: We'll evaluate Hit Rate and NDCG")

MATRIX FACTORIZATION WITH ALS

Preparing data...
   Matrix shape: 610 users × 9724 movies
   Fill rate: 1.70%

Using 20 latent factors

Training NMF model...
   (This may take a few seconds...)


C:\Users\HP\Desktop\mon_projet_personnel\recom_de_film\.venv\lib\site-packages\sklearn\decomposition\_nmf.py:1728: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(



   User factors matrix (W): 610 users × 20 factors
   Movie factors matrix (H): 20 factors × 9724 movies

Reconstructing rating matrix...
   Reconstructed matrix shape: (610, 9724)

Evaluating model...
   RMSE: 2.5346
   MAE:  2.1840

   Interpretation:
      - RMSE = average error of 2.53 stars
      - MAE = average absolute error of 2.18 stars

Example prediction:
   User 1, Movie 1: true=4.0, predicted=2.48

ALS MODEL COMPLETE

NEXT: We'll evaluate Hit Rate and NDCG


In [29]:
# Cellule 25 - Hit Rate et NDCG
# But: Évaluer la qualité des recommandations avec des métriques adaptées

import numpy as np
from sklearn.metrics import ndcg_score

print("="*50)
print("HIT RATE AND NDCG EVALUATION")
print("="*50)

# Étape 1: Définir la fonction Hit Rate
def hit_rate_at_k(y_true, y_pred, k=10, threshold=3.5):
    """
    Calcule le Hit Rate@k
    
    Paramètres:
    - y_true: matrice des vraies notes
    - y_pred: matrice des notes prédites
    - k: nombre de recommandations à considérer
    - threshold: seuil pour considérer un film comme "aimé"
    
    Retourne:
    - hit_rate: proportion d'utilisateurs avec au moins un film aimé dans le top k
    """
    hits = 0
    n_users = 0
    
    for user in range(len(y_true)):
        # Films que l'utilisateur a vraiment aimés (note > threshold)
        liked = np.where(y_true[user] > threshold)[0]
        
        if len(liked) == 0:
            continue  # Ignorer les utilisateurs sans films aimés
        
        n_users += 1
        
        # Prendre les K meilleures prédictions
        top_k = np.argsort(y_pred[user])[-k:][::-1]
        
        # Vérifier si au moins un film aimé est dans le top k
        if len(set(liked) & set(top_k)) > 0:
            hits += 1
    
    return hits / n_users if n_users > 0 else 0

# Étape 2: Calculer le Hit Rate
print("\nCalculating Hit Rate...")
hit_rate = hit_rate_at_k(R, R_pred, k=10, threshold=3.5)
print(f"   Hit Rate@10: {hit_rate:.4f}")
print(f"   Interpretation: {hit_rate*100:.1f}% of users have at least one liked movie in top 10")

# Étape 3: Calculer NDCG pour un exemple utilisateur
print("\nCalculating NDCG for sample user...")
user_idx = 0  # User 1
true_user = R[user_idx]
pred_user = R_pred[user_idx]

# Ne garder que les films que l'utilisateur a vraiment notés
mask = true_user > 0

if mask.sum() > 0:
    # Préparer les données pour NDCG
    true_masked = true_user[mask].reshape(1, -1)
    pred_masked = pred_user[mask].reshape(1, -1)
    
    # Calculer NDCG@10
    ndcg = ndcg_score(true_masked, pred_masked, k=min(10, mask.sum()))
    print(f"   User 1 NDCG@10: {ndcg:.4f}")
    print(f"   Interpretation: {ndcg:.2f} (1.0 = perfect ranking)")

print("\n" + "="*50)
print("EVALUATION COMPLETE")
print("="*50)
print("\nNEXT: We'll save models with MLFlow")

HIT RATE AND NDCG EVALUATION

Calculating Hit Rate...
   Hit Rate@10: 0.9343
   Interpretation: 93.4% of users have at least one liked movie in top 10

Calculating NDCG for sample user...
   User 1 NDCG@10: 0.9702
   Interpretation: 0.97 (1.0 = perfect ranking)

EVALUATION COMPLETE

NEXT: We'll save models with MLFlow


In [31]:
# Cellule 26 - Sauvegarder le modèle avec MLFlow
# But: Enregistrer le modèle ALS pour le réutiliser plus tard

import mlflow
import mlflow.sklearn
from datetime import datetime

print("="*50)
print("SAVING ALS MODEL WITH MLFLOW")
print("="*50)

# Démarrer une nouvelle expérience MLFlow
with mlflow.start_run(run_name="ALS_Final_Model"):
    
    # 1. Logger les paramètres
    print("\nLogging parameters...")
    mlflow.log_param("model_type", "ALS (NMF)")
    mlflow.log_param("n_factors", 20)
    mlflow.log_param("max_iter", 200)
    mlflow.log_param("alpha_W", 0.01)
    mlflow.log_param("alpha_H", 0.01)
    print("   Parameters logged")
    
    # 2. Logger les métriques (sans caractères spéciaux)
    print("\nLogging metrics...")
    mlflow.log_metric("RMSE", 2.5346)
    mlflow.log_metric("MAE", 2.1840)
    mlflow.log_metric("Hit_Rate_10", 0.9343)  # @ remplacé par _
    mlflow.log_metric("NDCG_10", 0.9702)      # @ remplacé par _
    print("   Metrics logged")
    
    # 3. Sauvegarder le modèle
    print("\nSaving model artifact...")
    mlflow.sklearn.log_model(model_nmf, "ALS_model")
    print("   Model saved")
    
    # 4. Ajouter des tags
    print("\nAdding tags...")
    mlflow.set_tag("dataset", "MovieLens")
    mlflow.set_tag("phase", "4.2")
    mlflow.set_tag("timestamp", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
    print("   Tags added")
    
    print("\nAll experiment data saved successfully")

print("\nTO VIEW SAVED MODEL:")
print("   1. In terminal, run: mlflow ui")
print("   2. Open http://localhost:5000 in your browser")
print("   3. Look for experiment: ALS_Final_Model")

print("\nMODEL STORED IN:")
print("   ./mlruns/")

print("\n" + "="*50)
print("MODEL SAVING COMPLETE")
print("="*50)

SAVING ALS MODEL WITH MLFLOW

Logging parameters...
   Parameters logged

Logging metrics...


2026/03/07 13:32:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


   Metrics logged

Saving model artifact...


2026/03/07 13:32:28 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/07 13:32:28 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!


   Model saved

Adding tags...
   Tags added

All experiment data saved successfully

TO VIEW SAVED MODEL:
   1. In terminal, run: mlflow ui
   2. Open http://localhost:5000 in your browser
   3. Look for experiment: ALS_Final_Model

MODEL STORED IN:
   ./mlruns/

MODEL SAVING COMPLETE
